# VPN Traffic Obfuscation - Comprehensive Analysis

## Overview
This notebook provides a detailed analysis of VPN traffic obfuscation techniques (Baseline WireGuard, UDP2RAW, OBFS4).
It analyzes data from four key sources:
1. **Metadata**: Run configurations, tool versions, and reproducibility seeds.
2. **Suricata**: Intrusion Detection System (IDS) alerts and signature matches.
3. **Zeek**: Network Security Monitor (NSM) flow logs and protocol detection.
4. **Packet Features**: Low-level packet characteristics (Size, Timing) extracted via Tshark.
5. **Performance**: Throughput metrics from iPerf3.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import glob
import os
import numpy as np

# Configure Aesthetics
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = [14, 8]
plt.rcParams['font.size'] = 12

RESULTS_DIR = "../results/runs"

def load_runs(base_dir):
    runs_data = []
    
    # Walk through results directory
    # Structure: results/runs/<scenario>/<timestamp>/
    # Or: results/runs/<scenario>/<mode>/<timestamp>/ depending on script
    # We search recursively for metadata.json
    
    meta_files = glob.glob(os.path.join(base_dir, "**", "metadata.json"), recursive=True)
    
    print(f"Found {len(meta_files)} runs with metadata.")
    
    for meta_path in meta_files:
        run_path = os.path.dirname(meta_path)
        
        # 1. Load Metadata
        try:
            with open(meta_path, 'r') as f:
                meta = json.load(f)
        except Exception as e:
            print(f"Skipping {run_path}: Invalid metadata ({e})")
            continue
            
        run_data = {
            "id": os.path.basename(run_path),
            "path": run_path,
            "scenario": meta.get("scenario", "unknown"),
            "mode": meta.get("mode", "unknown"),
            "seed": meta.get("seed", "N/A"),
            "timestamp": meta.get("timestamp", "N/A"),
            "suricata_version": meta.get("versions", {}).get("suricata", "N/A"),
            "zeek_version": meta.get("versions", {}).get("zeek", "N/A"),
            "tshark_version": meta.get("versions", {}).get("tshark", "N/A"),
            "suricata_rules_count": meta.get("loaded_rules_count", 0)
        }
        
        # 2. Load Suricata Alerts (EVE JSON)
        eve_path = os.path.join(run_path, "suricata", "eve.json")
        alerts = []
        if os.path.exists(eve_path):
            try:
                # Read line by line as it is NDJSON
                with open(eve_path, 'r') as f:
                    for line in f:
                        entry = json.loads(line)
                        if entry.get('event_type') == 'alert':
                            alerts.append(entry.get('alert'))
            except Exception as e: 
                print(f"Error reading eve.json in {run_path}: {e}")
        run_data["alerts"] = pd.DataFrame(alerts)
        
        # 3. Load Zeek Flows (Conn.log JSON)
        conn_path = os.path.join(run_path, "zeek", "conn.log")
        zeek_flows = []
        if os.path.exists(conn_path):
            try:
                # Determine if it's JSON or TSV
                # New setup is JSON
                with open(conn_path, 'r') as f:
                    first_char = f.read(1)
                
                if first_char == '{':
                    # JSON Log
                    with open(conn_path, 'r') as f:
                        for line in f:
                            try:
                                zeek_flows.append(json.loads(line))
                            except: pass
                    run_data["zeek_df"] = pd.DataFrame(zeek_flows)
                else:
                    # Legacy/Fallback TSV
                    run_data["zeek_df"] = pd.read_csv(conn_path, sep="\t", comment="#", on_bad_lines='skip')
            except Exception as e:
                print(f"Error reading zeek conn.log in {run_path}: {e}")
                run_data["zeek_df"] = pd.DataFrame()
        else:
            run_data["zeek_df"] = pd.DataFrame()
            
        # 4. Load Packet Features (CSV)
        packet_path = os.path.join(run_path, "pcap_features", "packets.csv")
        if os.path.exists(packet_path):
            try:
                run_data["packets_df"] = pd.read_csv(packet_path)
                # Ensure numeric types
                run_data["packets_df"]['frame.len'] = pd.to_numeric(run_data["packets_df"]['frame.len'], errors='coerce')
                run_data["packets_df"]['frame.time_epoch'] = pd.to_numeric(run_data["packets_df"]['frame.time_epoch'], errors='coerce')
            except Exception as e:
                print(f"Error reading packets.csv in {run_path}: {e}")
                run_data["packets_df"] = pd.DataFrame()
        else:
            run_data["packets_df"] = pd.DataFrame()

        # 5. Load Performance (iPerf JSON)
        iperf_path = os.path.join(run_path, "iperf", "iperf.json")
        run_data["throughput_mbps"] = 0.0
        run_data["retransmits"] = 0
        
        if os.path.exists(iperf_path):
            try:
                with open(iperf_path, 'r') as f:
                    iperf_data = json.load(f)
                    # Extract summary
                    end = iperf_data.get('end', {})
                    sum_sent = end.get('sum_sent', {})
                    bps = sum_sent.get('bits_per_second', 0)
                    run_data["throughput_mbps"] = bps / 1e6
                    run_data["retransmits"] = sum_sent.get('retransmits', 0)
            except Exception as e:
                # Might be empty or invalid if burst mode
                pass

        runs_data.append(run_data)
        
    return pd.DataFrame(runs_data)

df_runs = load_runs(RESULTS_DIR)

# Enrich Label
if not df_runs.empty:
    df_runs['label'] = df_runs['scenario'] + " (" + df_runs['mode'] + ")"
    
    # Normalize columns if missing
    if 'packets_df' not in df_runs.columns:
        df_runs['packets_df'] = None
    if 'zeek_df' not in df_runs.columns:
        df_runs['zeek_df'] = None

    print(f"Successfully loaded {len(df_runs)} runs.")
    display(df_runs[['id', 'scenario', 'mode', 'timestamp', 'seed', 'suricata_rules_count']].sort_values('timestamp', ascending=False))
else:
    print("No valid runs found. Please execute experiment scripts first.")

## 2. IDS Visibility (Suricata & Zeek)
How well do the security tools see the traffic?
- **Suricata Alerts**: Normalized per minute.
- **Zeek Protocol Detection**: Percentage of flows identified as 'unknown' or 'SSL/TLS'.

In [ ]:
if not df_runs.empty:
    # Aggregate Alerts
    alert_stats = []
    for _, row in df_runs.iterrows():
        alerts_df = row['alerts']
        count = len(alerts_df) if not alerts_df.empty else 0
        
        # Determine duration for normalization
        duration = 60 # Default assumption
        if row['zeek_df'] is not None and not row['zeek_df'].empty:
             # Try to get duration from Zeek logs sum
             try:
                 zeek_durs = pd.to_numeric(row['zeek_df']['duration'], errors='coerce').sum()
                 if zeek_durs > 0: duration = max(duration, zeek_durs)
             except: pass
        
        alert_stats.append({
            'label': row['label'],
            'total_alerts': count,
            'alerts_per_min': (count / duration) * 60
        })
    
    df_alerts = pd.DataFrame(alert_stats)
    
    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.barplot(data=df_alerts, x='label', y='total_alerts', hue='label', ax=ax[0], palette='viridis')
    ax[0].set_title("Total Suricata Alerts")
    ax[0].tick_params(axis='x', rotation=45)

    # Zeek Protocol Breakdown
    zeek_proto_stats = []
    for _, row in df_runs.iterrows():
        zdf = row['zeek_df']
        if zdf is not None and not zdf.empty and 'service' in zdf.columns:
            # Count services
            # Fill NA or '-' with 'unknown'
            services = zdf['service'].replace('-', 'unknown').fillna('unknown')
            counts = services.value_counts(normalize=True).reset_index()
            counts.columns = ['service', 'ratio']
            counts['label'] = row['label']
            zeek_proto_stats.append(counts)
    
    if zeek_proto_stats:
        df_zeek = pd.concat(zeek_proto_stats)
        sns.barplot(data=df_zeek, x='label', y='ratio', hue='service', ax=ax[1])
        ax[1].set_title("Zeek Detected Protocols (Ratio)")
        ax[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 3. Traffic Fingerprinting (Packet-Level)
Analysis of packet sizes and Inter-Arrival Times (IAT) to identify obfuscation patterns.
We separate TCP and UDP traffic.

In [ ]:
def calculate_iat(df):
    if df.empty or 'frame.time_epoch' not in df.columns: return []
    # Sort by time
    df = df.sort_values('frame.time_epoch')
    # Calculate diff
    iat = df['frame.time_epoch'].diff().dropna()
    return iat[iat > 0] * 1000 # Convert to ms

if not df_runs.empty:
    # Filter only runs with packet data
    valid_runs = df_runs[df_runs['packets_df'].notnull()]
    
    if not valid_runs.empty:
        # PACKET SIZES
        plt.figure(figsize=(12, 6))
        for _, row in valid_runs.iterrows():
            pdf = row['packets_df']
            if pdf.empty: continue
            
            # Filter by protocol if needed (e.g. only UDP for WireGuard)
            # WireGuard is UDP (17), TCP is 6
            # We plot distribution of all packets for now to see overhead
            sns.kdeplot(pdf['frame.len'], label=row['label'], fill=False, linewidth=2)
            
        plt.title("Packet Size Distribution (KDE)")
        plt.xlabel("Packet Size (Bytes)")
        plt.xlim(0, 1600)
        plt.legend()
        plt.show()
        
        # IAT Analysis (Boxplot)
        iat_data = []
        for _, row in valid_runs.iterrows():
            pdf = row['packets_df']
            if pdf.empty: continue
            
            iat = calculate_iat(pdf)
            # Downsample for plotting if huge
            if len(iat) > 1000: iat = iat.sample(1000, random_state=42)
            
            temp_df = pd.DataFrame({'iat_ms': iat})
            temp_df['label'] = row['label']
            iat_data.append(temp_df)
            
        if iat_data:
            df_iat = pd.concat(iat_data)
            plt.figure(figsize=(12, 6))
            sns.boxplot(data=df_iat, x='label', y='iat_ms', showfliers=False)
            plt.title("Inter-Arrival Time (IAT) Distribution (Outliers Hidden)")
            plt.ylabel("IAT (ms)")
            plt.xticks(rotation=45)
            plt.show()

## 4. Protocol Plausibility Check
Does traffic on TCP/443 actually look like TLS? 
We check for `tls.handshake.type == 1` (Client Hello) in flows on port 443.

In [ ]:
if not df_runs.empty:
    plausibility_stats = []
    
    for _, row in valid_runs.iterrows():
        pdf = row['packets_df']
        if pdf.empty: continue
        
        # Check if 'tls.handshake.type' exists
        if 'tls.handshake.type' not in pdf.columns:
            continue
            
        # Filter potentially interesting traffic (e.g. Dst Port 443)
        # We need numeric ports
        pdf['tcp.dstport'] = pd.to_numeric(pdf['tcp.dstport'], errors='coerce')
        
        # Packets on 443
        https_packets = pdf[pdf['tcp.dstport'] == 443]
        total_443 = len(https_packets)
        
        if total_443 == 0:
            # Maybe it's udp2raw on UDP?
            pass
        else:
            # Count Client Hellos (Type 1)
            client_hellos = len(https_packets[https_packets['tls.handshake.type'] == 1])
            
            plausibility_stats.append({
                'label': row['label'],
                'total_packets_443': total_443,
                'client_hellos': client_hellos,
                'ratio_hello': client_hellos / total_443 if total_443 > 0 else 0
            })
    
    if plausibility_stats:
        df_plaus = pd.DataFrame(plausibility_stats)
        print("Protocol Plausibility on Port 443:")
        display(df_plaus)
        
        plt.figure(figsize=(8, 5))
        sns.barplot(data=df_plaus, x='label', y='ratio_hello', palette='magma')
        plt.title("Ratio of Start-of-Flows containing TLS Client Hello (Port 443)")
        plt.ylabel("Client Hello Ratio (approx per packet)")
        plt.show()

## 5. Performance Impact
Throughput comparison and relative overhead.

In [ ]:
if not df_runs.empty:
    # Filter Streaming modes usually have valid iperf data
    perf_df = df_runs[(df_runs['throughput_mbps'] > 0)].copy()
    
    if not perf_df.empty:
        # Calculate Overhead vs Baseline
        # Find baseline throughput
        baseline_row = perf_df[perf_df['scenario'] == 'baseline']
        if not baseline_row.empty:
            base_tp = baseline_row['throughput_mbps'].mean()
            perf_df['overhead_pct'] = ((base_tp - perf_df['throughput_mbps']) / base_tp) * 100
        else:
            perf_df['overhead_pct'] = 0
            
        fig, ax = plt.subplots(1, 2, figsize=(14, 5))
        
        sns.barplot(data=perf_df, x='label', y='throughput_mbps', ax=ax[0], palette='coolwarm')
        ax[0].set_title("Throughput (Mbps)")
        
        sns.barplot(data=perf_df, x='label', y='overhead_pct', ax=ax[1], palette='Reds')
        ax[1].set_title("Relative Overhead (% cost vs Baseline)")
        
        plt.tight_layout()
        plt.show()
    else:
        print("No valid iPerf3 throughput data found (did you run 'streaming' mode?)")